# Milestone 7: Production AI Copilot from scratch
*AI from scratch, milestone 7* · capstone combine notebook for Stage 7

Build a complete production AI Copilot ("Personal Second-Brain & Incident Partner") that ties together every pillar of the curriculum:
1. **Data Engineering with SQL**: Ingest documents into an SQLite relational store and use window functions to retrieve point-in-time chunk states without leakage.
2. **Multimodal Voice & Receipt OCR Ingestion**: Clean Whisper voice-memo transcripts and parse structured fields from receipt OCR scans before indexing.
3. **Classic Tabular Classifier & Cost Matrix**: Predict document urgency and priority using metadata, tuning the decision threshold against asymmetric false-negative vs false-positive mistake costs.
4. **PyTorch Neural Bi-Encoder**: Implement a custom neural projection head in PyTorch that maps text vectors to unit-normalized embeddings, and build cosine vector search.
5. **2D SVD Semantic Topic Map**: Project high-dimensional TF-IDF vectors onto their top 2 right singular vectors so the copilot can render an interactive 2D topic map.
6. **Grounded RAG with Strict Provenance**: Format multi-chunk contexts with explicit `[doc#chunk]` citation brackets.
7. **Automated Grounding Evaluation**: Quantify the citation grounding ratio mathematically to catch hallucinations before reaching users.
8. **Production Serving & MLOps Drift**: Wrap the pipeline in a FastAPI-compatible service handler with a 100 ms latency budget, and compute the Population Stability Index (PSI) to detect distribution drift.

Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it; `hint(n)` and `answer(n)`
are there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['Start by partitioning rows per document: SELECT doc_id, chunk_id, text, ROW_NUMBER() OVER (PARTITION BY doc_id ORDER BY chunk_seq ASC) as rn FROM chunks', 'Filter the outer query with WHERE rn = 1 and call cur.execute(query).fetchall()'], 2: ['For "voice", chain the two re.sub(...) calls, check whether any incident keyword is in cleaned.lower(), and count cleaned.split().', 'For "receipt", extract vendor, item, and total with re.search(...), build the summary string, and return the dict including "total_usd".'], 3: ['Fit LogisticRegression().fit(X, y) and take column 1 of predict_proba(X).', 'Loop over np.linspace(0.1, 0.9, 9), compute fn and fp at each threshold, and keep the threshold with lowest 10 * fn + fp.'], 4: ['Subclass nn.Module, create self.proj = nn.Linear(in_features, out_features, bias=False) in __init__.', 'In forward(self, x), return F.normalize(self.proj(x), p=2, dim=-1).'], 5: ['Center the columns first: X_c = tfidf_matrix - np.mean(tfidf_matrix, axis=0, keepdims=True), then call np.linalg.svd(X_c, full_matrices=False).', 'Multiply X_c @ Vt[:2].T for the (N, 2) coordinates, and divide np.sum(S[:2] ** 2) by np.sum(S ** 2).'], 6: ['Transform [query_str] with tfidf, wrap in torch.tensor(..., dtype=torch.float32), and pass through model(q_tensor).', 'Compute sims = torch.mm(q_emb, doc_embeddings.T)[0] and return [int(idx) for idx in torch.argsort(-sims)[:top_k].numpy()].'], 7: ['For each index i in retrieved_indices, format f"[{chunks_data[i][1]}#{chunks_data[i][4]}] {chunks_data[i][2]}".', 'Join the formatted lines with "\\n".join(lines).'], 8: ['Tokenize lowercase alphanumeric words longer than 3 characters into sets a_toks and c_toks.', 'Return len(a_toks & c_toks) / len(a_toks) if a_toks is non-empty, else 0.0.'], 9: ['Score urgency probability p with clf.predict_proba([request_dict["metadata"]])[0, 1] and retrieve hits = retrieve_top_k(request_dict["query"], top_k=2).', 'Format citations as [f"{chunks_data[i][1]}#{chunks_data[i][4]}" for i in hits] and measure latency_ms = (time.perf_counter() - t0) * 1000.'], 10: ['Bin both arrays over range=(0.0, 1.0) with np.histogram, add 1e-4 to counts, and divide each by its sum to get p and q.', 'Return float(np.sum((p - q) * np.log(p / q))).']}
_ANSWERS = {1: 'cXVlcnkgPSAiIiIKU0VMRUNUIGRvY19pZCwgY2h1bmtfaWQsIHRleHQKRlJPTSAoCiAgICBTRUxFQ1QgZG9jX2lkLCBjaHVua19pZCwgdGV4dCwKICAgICAgICAgICBST1dfTlVNQkVSKCkgT1ZFUiAoUEFSVElUSU9OIEJZIGRvY19pZCBPUkRFUiBCWSBjaHVua19zZXEgQVNDKSBhcyBybgogICAgRlJPTSBjaHVua3MKKSBXSEVSRSBybiA9IDEKIiIiCmZpcnN0X2NodW5rcyA9IGN1ci5leGVjdXRlKHF1ZXJ5KS5mZXRjaGFsbCgp', 2: 'ZGVmIHBhcnNlX211bHRpbW9kYWxfaXRlbShyYXdfdGV4dCwgc291cmNlX2tpbmQpOgogICAgaWYgc291cmNlX2tpbmQgPT0gInZvaWNlIjoKICAgICAgICBjbGVhbmVkID0gcmUuc3ViKHIiXFtcZHsyfTpcZHsyfVxdXHMqIiwgIiIsIHJhd190ZXh0KQogICAgICAgIGNsZWFuZWQgPSByZS5zdWIociJcYih1aG18dWgpXGIsP1xzKiIsICIiLCBjbGVhbmVkLCBmbGFncz1yZS5JKS5zdHJpcCgpCiAgICAgICAgaXNfaW5jID0gaW50KGFueSh3IGluIGNsZWFuZWQubG93ZXIoKSBmb3IgdyBpbiAoImZhaWxlZCIsICJvdXRhZ2UiLCAiZXhoYXVzdGVkIiwgImxlYWsiKSkpCiAgICAgICAgcmV0dXJuIHsidGV4dCI6IGNsZWFuZWQsICJpc19pbmNpZGVudCI6IGlzX2luYywgIndvcmRfY291bnQiOiBsZW4oY2xlYW5lZC5zcGxpdCgpKX0KICAgIHZlbmRvciA9IHJlLnNlYXJjaChyIlZFTkRPUjpccyooW158XSspIiwgcmF3X3RleHQpLmdyb3VwKDEpLnN0cmlwKCkKICAgIGl0ZW0gPSByZS5zZWFyY2gociJJVEVNOlxzKihbXnxdKykiLCByYXdfdGV4dCkuZ3JvdXAoMSkuc3RyaXAoKQogICAgdG90YWwgPSBmbG9hdChyZS5zZWFyY2gociJUT1RBTDpccypcJD8oWzAtOS5dKykiLCByYXdfdGV4dCkuZ3JvdXAoMSkpCiAgICBzdW1tYXJ5ID0gZiJSZWNlaXB0IGZyb20ge3ZlbmRvcn06IHtpdGVtfSB0b3RhbGluZyAke3RvdGFsOi4yZn0uIgogICAgcmV0dXJuIHsidGV4dCI6IHN1bW1hcnksICJpc19pbmNpZGVudCI6IDAsICJ3b3JkX2NvdW50IjogbGVuKHN1bW1hcnkuc3BsaXQoKSksICJ0b3RhbF91c2QiOiB0b3RhbH0=', 3: 'Y2xmID0gTG9naXN0aWNSZWdyZXNzaW9uKCkuZml0KFgsIHkpCnByb2JzID0gY2xmLnByZWRpY3RfcHJvYmEoWClbOiwgMV0KCnRocmVzaG9sZHMgPSBucC5saW5zcGFjZSgwLjEsIDAuOSwgOSkKYmVzdF9jb3N0ID0gZmxvYXQoImluZiIpCmJlc3RfdGhyZXNob2xkID0gMC41Cgpmb3IgdGggaW4gdGhyZXNob2xkczoKICAgIHByZWRzID0gKHByb2JzID49IHRoKS5hc3R5cGUoaW50KQogICAgZm4gPSBucC5zdW0oKHByZWRzID09IDApICYgKHkgPT0gMSkpCiAgICBmcCA9IG5wLnN1bSgocHJlZHMgPT0gMSkgJiAoeSA9PSAwKSkKICAgIGNvc3QgPSAxMCAqIGZuICsgZnAKICAgIGlmIGNvc3QgPCBiZXN0X2Nvc3Q6CiAgICAgICAgYmVzdF9jb3N0ID0gY29zdAogICAgICAgIGJlc3RfdGhyZXNob2xkID0gZmxvYXQodGgp', 4: 'Y2xhc3MgQmlFbmNvZGVySGVhZChubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2ZlYXR1cmVzLCBvdXRfZmVhdHVyZXM9MTYpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYucHJvaiA9IG5uLkxpbmVhcihpbl9mZWF0dXJlcywgb3V0X2ZlYXR1cmVzLCBiaWFzPUZhbHNlKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHJldHVybiBGLm5vcm1hbGl6ZShzZWxmLnByb2ooeCksIHA9MiwgZGltPS0xKQoKdG9yY2gubWFudWFsX3NlZWQoNDIpCm1vZGVsID0gQmlFbmNvZGVySGVhZCh2b2NhYl9zaXplLCAxNikKd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICBkb2NfZW1iZWRkaW5ncyA9IG1vZGVsKERfdGVuc29yKQ==', 5: 'ZGVmIHByb2plY3RfY2h1bmtzXzJkKHRmaWRmX21hdHJpeCk6CiAgICBYX2MgPSB0ZmlkZl9tYXRyaXggLSBucC5tZWFuKHRmaWRmX21hdHJpeCwgYXhpcz0wLCBrZWVwZGltcz1UcnVlKQogICAgVSwgUywgVnQgPSBucC5saW5hbGcuc3ZkKFhfYywgZnVsbF9tYXRyaWNlcz1GYWxzZSkKICAgIGNvb3Jkc18yZCA9IFhfYyBAIFZ0WzoyXS5UCiAgICB2YXJfcmF0aW8gPSBmbG9hdChucC5zdW0oU1s6Ml0gKiogMikgLyBucC5zdW0oUyAqKiAyKSkKICAgIHJldHVybiBjb29yZHNfMmQsIHZhcl9yYXRpbw==', 6: 'ZGVmIHJldHJpZXZlX3RvcF9rKHF1ZXJ5X3N0ciwgdG9wX2s9Myk6CiAgICBxX3RlbnNvciA9IHRvcmNoLnRlbnNvcih0ZmlkZi50cmFuc2Zvcm0oW3F1ZXJ5X3N0cl0pLnRvYXJyYXkoKSwgZHR5cGU9dG9yY2guZmxvYXQzMikKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIHFfZW1iID0gbW9kZWwocV90ZW5zb3IpCiAgICAgICAgc2ltcyA9IHRvcmNoLm1tKHFfZW1iLCBkb2NfZW1iZWRkaW5ncy5UKVswXQogICAgcmV0dXJuIFtpbnQoaWR4KSBmb3IgaWR4IGluIHRvcmNoLmFyZ3NvcnQoLXNpbXMpWzp0b3Bfa10ubnVtcHkoKV0=', 7: 'ZGVmIGJ1aWxkX2dyb3VuZGVkX2NvbnRleHQocmV0cmlldmVkX2luZGljZXMpOgogICAgbGluZXMgPSBbZiJbe2NodW5rc19kYXRhW2ldWzFdfSN7Y2h1bmtzX2RhdGFbaV1bNF19XSB7Y2h1bmtzX2RhdGFbaV1bMl19IiBmb3IgaSBpbiByZXRyaWV2ZWRfaW5kaWNlc10KICAgIHJldHVybiAiXG4iLmpvaW4obGluZXMp', 8: 'ZGVmIGNvbXB1dGVfZ3JvdW5kaW5nX3JhdGlvKGFuc3dlcl9zZW50ZW5jZSwgY29udGV4dF90ZXh0KToKICAgIGRlZiBjbGVhbihzKToKICAgICAgICByZXR1cm4gW3cgZm9yIHcgaW4gIiIuam9pbihjIGlmIGMuaXNhbG51bSgpIGVsc2UgIiAiIGZvciBjIGluIHMubG93ZXIoKSkuc3BsaXQoKSBpZiBsZW4odykgPiAzXQogICAgYV90b2tzID0gc2V0KGNsZWFuKGFuc3dlcl9zZW50ZW5jZSkpCiAgICBjX3Rva3MgPSBzZXQoY2xlYW4oY29udGV4dF90ZXh0KSkKICAgIGlmIG5vdCBhX3Rva3M6CiAgICAgICAgcmV0dXJuIDAuMAogICAgcmV0dXJuIGZsb2F0KGxlbihhX3Rva3MuaW50ZXJzZWN0aW9uKGNfdG9rcykpIC8gbGVuKGFfdG9rcykp', 9: 'ZGVmIGNvcGlsb3Rfc2VydmljZShyZXF1ZXN0X2RpY3QpOgogICAgdDAgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICBwID0gZmxvYXQoY2xmLnByZWRpY3RfcHJvYmEoW3JlcXVlc3RfZGljdFsibWV0YWRhdGEiXV0pWzAsIDFdKQogICAgaGl0cyA9IHJldHJpZXZlX3RvcF9rKHJlcXVlc3RfZGljdFsicXVlcnkiXSwgdG9wX2s9MikKICAgIGNpdHMgPSBbZiJ7Y2h1bmtzX2RhdGFbaV1bMV19I3tjaHVua3NfZGF0YVtpXVs0XX0iIGZvciBpIGluIGhpdHNdCiAgICBsYXRlbmN5X21zID0gKHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MCkgKiAxMDAwLjAKICAgIHJldHVybiB7CiAgICAgICAgImNpdGF0aW9ucyI6IGNpdHMsCiAgICAgICAgImlzX3VyZ2VudCI6IGJvb2wocCA+PSBiZXN0X3RocmVzaG9sZCksCiAgICAgICAgInVyZ2VuY3lfc2NvcmUiOiBwLAogICAgICAgICJsYXRlbmN5X21zIjogbGF0ZW5jeV9tcwogICAgfQ==', 10: 'ZGVmIGNhbGN1bGF0ZV9wc2koYmFzZWxpbmVfc2NvcmVzLCBwcm9kdWN0aW9uX3Njb3JlcywgbnVtX2JpbnM9NCk6CiAgICBwX2NvdW50cywgYmlucyA9IG5wLmhpc3RvZ3JhbShiYXNlbGluZV9zY29yZXMsIGJpbnM9bnVtX2JpbnMsIHJhbmdlPSgwLjAsIDEuMCkpCiAgICBxX2NvdW50cywgXyA9IG5wLmhpc3RvZ3JhbShwcm9kdWN0aW9uX3Njb3JlcywgYmlucz1iaW5zKQogICAgcCA9IChwX2NvdW50cyArIDFlLTQpIC8gbnAuc3VtKHBfY291bnRzICsgMWUtNCkKICAgIHEgPSAocV9jb3VudHMgKyAxZS00KSAvIG5wLnN1bShxX2NvdW50cyArIDFlLTQpCiAgICByZXR1cm4gZmxvYXQobnAuc3VtKChwIC0gcSkgKiBucC5sb2cocCAvIHEpKSk='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import math, time, sqlite3, json, re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

print(f"PyTorch version: {torch.__version__} · SQLite3 version: {sqlite3.sqlite_version}")

## 1. Data engineering & SQL window functions
Our copilot ingests technical documents, voice memos, receipts, and incident reports.
An in-memory SQLite database has been initialized below with two tables:
- `documents`: `doc_id TEXT PRIMARY KEY, title TEXT, is_incident INT, created_hour INT, is_urgent INT`
- `chunks`: `chunk_id TEXT PRIMARY KEY, doc_id TEXT, text TEXT, word_count INT, chunk_seq INT`

Write a SQL query using the window function `ROW_NUMBER() OVER (PARTITION BY doc_id ORDER BY chunk_seq ASC)`
to extract the first chunk (`chunk_seq = 1`) of every document.
Fetch and store the resulting list of tuples `[(doc_id, chunk_id, text), ...]` in `first_chunks`.

In [ ]:
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE documents (doc_id TEXT PRIMARY KEY, title TEXT, is_incident INT, created_hour INT, is_urgent INT)")
cur.execute("CREATE TABLE chunks (chunk_id TEXT PRIMARY KEY, doc_id TEXT, text TEXT, word_count INT, chunk_seq INT)")

docs_data = [
    ("doc-1", "Transformer Attention Guide", 0, 14, 0),
    ("doc-2", "API Gateway Outage Report", 1, 9, 1),
    ("doc-3", "Optimizer Convergence Notes", 0, 11, 0),
    ("doc-4", "Database Pool Exhaustion", 1, 8, 1),
    ("doc-5", "LoRA Adapter Training", 0, 15, 0),
]
cur.executemany("INSERT INTO documents VALUES (?,?,?,?,?)", docs_data)

chunks_data = [
    ("c-1-1", "doc-1", "Self-attention computes dynamic weights from query and key dot products.", 11, 1),
    ("c-1-2", "doc-1", "Multi-head attention projects vectors into multiple representation subspaces.", 8, 2),
    ("c-2-1", "doc-2", "Production router failed due to memory exhaustion under spike load.", 10, 1),
    ("c-3-1", "doc-3", "Learning rates exceeding two over L cause oscillations in steep valleys.", 11, 1),
    ("c-4-1", "doc-4", "Backend database pool exhausted after connection leak in service worker.", 10, 1),
    ("c-5-1", "doc-5", "LoRA freezes base model weights and fine-tunes low rank adapter matrices.", 12, 1),
]
cur.executemany("INSERT INTO chunks VALUES (?,?,?,?,?)", chunks_data)
conn.commit()

# YOUR CODE HERE (stuck? hint(1), then answer(1))
...

if "first_chunks" in locals() and isinstance(first_chunks, list):
    print(f"Extracted {len(first_chunks)} first-chunks using SQL window partition.")

In [ ]:
check(1, lambda: len(first_chunks) == 5 and all(len(row) == 3 for row in first_chunks), "SQL window function extracts exactly one starting chunk per document without duplicates")

## 2. Multimodal ingestion: Whisper voice transcripts & receipt OCR
A real second-brain copilot ingests more than clean markdown: it receives raw Whisper audio transcripts
with `[MM:SS]` timestamps and spoken fillers (`uhm`, `uh`), plus OCR text from receipts (`VENDOR: ... | ITEM: ... | TOTAL: $...`).

Write `parse_multimodal_item(raw_text, source_kind)`:
- When `source_kind == "voice"`:
  1. Strip `[MM:SS]` timestamps using `re.sub(r"\[\d{2}:\d{2}\]\s*", "", raw_text)`.
  2. Strip filler words `uhm` and `uh` using `re.sub(r"\b(uhm|uh)\b,?\s*", "", text, flags=re.I).strip()`.
  3. Set `is_incident = int(any(w in cleaned.lower() for w in ("failed", "outage", "exhausted", "leak")))`.
  4. Return `{"text": cleaned, "is_incident": is_incident, "word_count": len(cleaned.split())}`.
- When `source_kind == "receipt"`:
  1. Extract `vendor = re.search(r"VENDOR:\s*([^|]+)", raw_text).group(1).strip()`.
  2. Extract `item = re.search(r"ITEM:\s*([^|]+)", raw_text).group(1).strip()`.
  3. Extract `total = float(re.search(r"TOTAL:\s*\$?([0-9.]+)", raw_text).group(1))`.
  4. Format `summary = f"Receipt from {vendor}: {item} totaling ${total:.2f}."` and return
     `{"text": summary, "is_incident": 0, "word_count": len(summary.split()), "total_usd": total}`.

In [ ]:
sample_voice = "[00:01] Uhm, production router failed due to memory exhaustion under spike load."
sample_receipt = "VENDOR: CloudGPU Inc | DATE: 2026-10-20 | ITEM: 8x H100 Spot Node | TOTAL: $412.50"

# YOUR CODE HERE (stuck? hint(2), then answer(2))
...

if "parse_multimodal_item" in locals() and callable(parse_multimodal_item):
    try:
        print("Voice:", parse_multimodal_item(sample_voice, "voice"))
        print("Receipt:", parse_multimodal_item(sample_receipt, "receipt"))
    except Exception:
        pass

In [ ]:
check(2, lambda: parse_multimodal_item(sample_voice, "voice")["is_incident"] == 1 and "[00:01]" not in parse_multimodal_item(sample_voice, "voice")["text"] and abs(parse_multimodal_item(sample_receipt, "receipt")["total_usd"] - 412.50) < 1e-6, "multimodal parser cleans Whisper voice transcripts and extracts structured receipt totals")

## 3. Tabular metadata classifier & asymmetric cost-aware threshold
Before running expensive RAG search, the copilot triages incoming documents: is this an urgent incident?
Features: `[word_count, is_incident, created_hour]`.
1. Fit a `LogisticRegression` on `X` and labels `y = [0, 1, 0, 1, 0]`.
2. Get predicted positive-class probabilities `probs = clf.predict_proba(X)[:, 1]`.
3. In production, missing an urgent incident (False Negative) costs **10x** more than a false alarm (False Positive):
   $\text{Cost} = 10 \cdot \text{FN} + 1 \cdot \text{FP}$.
4. Search across candidate thresholds `np.linspace(0.1, 0.9, 9)`. Find the threshold that minimizes total cost,
   and assign it to `best_threshold`.

In [ ]:
X = np.array([
    [11, 0, 14],  # doc-1: non-incident
    [10, 1, 9],   # doc-2: incident
    [11, 0, 11],  # doc-3: non-incident
    [10, 1, 8],   # doc-4: incident
    [12, 0, 15]   # doc-5: non-incident
])
y = np.array([0, 1, 0, 1, 0])

# YOUR CODE HERE (stuck? hint(3), then answer(3))
...

if "best_threshold" in locals() and isinstance(best_threshold, float):
    print(f"Optimal cost-aware decision threshold: {best_threshold:.2f} (Total Cost: {best_cost})")

In [ ]:
check(3, lambda: best_threshold <= 0.5 and best_cost == 0, "asymmetric threshold prevents costly false negatives (missing an urgent alert)")

## 4. PyTorch Neural Bi-Encoder projection head
Implement a PyTorch `nn.Module` class named `BiEncoderHead`:
- Has a linear layer `self.proj = nn.Linear(in_features, out_features, bias=False)`.
- In `forward(x)`, passes `x` through `self.proj` and returns the L2-normalized vector `F.normalize(..., p=2, dim=-1)`.

Instantiate `model = BiEncoderHead(vocab_size, 16)` and project `D_tensor` into `doc_embeddings`.

In [ ]:
all_chunk_texts = [row[2] for row in chunks_data]
tfidf = TfidfVectorizer().fit(all_chunk_texts)
vocab_size = len(tfidf.get_feature_names_out())
D_tensor = torch.tensor(tfidf.transform(all_chunk_texts).toarray(), dtype=torch.float32)

# YOUR CODE HERE (stuck? hint(4), then answer(4))
...

if "doc_embeddings" in locals() and isinstance(doc_embeddings, torch.Tensor):
    print(f"Projected chunks shape: {doc_embeddings.shape}")

In [ ]:
check(4, lambda: doc_embeddings.shape == (6, 16) and torch.allclose(torch.norm(doc_embeddings, dim=-1), torch.ones(6)), "PyTorch bi-encoder projects document chunks into unit-normalized 16D embedding vectors")

## 5. 2D SVD semantic topic map projection
To visualize the knowledge base in the browser, the copilot projects the high-dimensional TF-IDF matrix onto 2D
using Singular Value Decomposition ($\mathbf{X}_c = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^\top$, from L3 SVD):
Write `project_chunks_2d(tfidf_matrix)` that:
1. Centers the matrix by subtracting the column mean: `X_c = tfidf_matrix - np.mean(tfidf_matrix, axis=0, keepdims=True)`.
2. Computes `U, S, Vt = np.linalg.svd(X_c, full_matrices=False)`.
3. Projects onto the first 2 right singular vectors: `coords_2d = X_c @ Vt[:2].T`.
4. Computes the fraction of variance captured by the first 2 singular values: `var_ratio = float(np.sum(S[:2] ** 2) / np.sum(S ** 2))`.
Return `(coords_2d, var_ratio)`.

In [ ]:
X_tfidf_np = D_tensor.numpy()

# YOUR CODE HERE (stuck? hint(5), then answer(5))
...

if "project_chunks_2d" in locals() and callable(project_chunks_2d):
    try:
        coords_2d, svd_var = project_chunks_2d(X_tfidf_np)
        print(f"2D SVD shape: {coords_2d.shape} · Variance explained by top 2 axes: {svd_var:.3f}")
    except Exception:
        pass

In [ ]:
check(5, lambda: project_chunks_2d(X_tfidf_np)[0].shape == (6, 2) and abs(project_chunks_2d(X_tfidf_np)[1] - 0.4323) < 0.01, "SVD projects the 6 chunks into 2D coordinates capturing 43.2% of total variance")

## 6. Cosine vector retrieval
Write `retrieve_top_k(query_str, top_k=3)` that:
1. Transforms `query_str` into TF-IDF vector, converts to PyTorch float tensor `q_tensor`.
2. Computes query embedding `q_emb = model(q_tensor)`.
3. Computes cosine similarities with all chunks via matrix multiplication: `sims = torch.mm(q_emb, doc_embeddings.T)[0]`.
4. Returns the top `top_k` indices sorted descending by similarity as a list of integers.

In [ ]:
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...

if "retrieve_top_k" in locals() and callable(retrieve_top_k):
    try:
        top_hits = retrieve_top_k("memory router load spike", top_k=3)
        print(f"Top-3 retrieved chunk indices for incident query: {top_hits}")
    except Exception:
        top_hits = None

In [ ]:
check(6, lambda: len(retrieve_top_k("memory router load spike", top_k=3)) == 3 and retrieve_top_k("memory router load spike", top_k=3)[0] == 2, "vector search retrieves the router memory outage chunk as the top match for memory spike query")

## 7. Grounded RAG context assembly with citation provenance
Write `build_grounded_context(retrieved_indices)` that accepts a list of chunk indices and builds
a context string where each retrieved chunk is tagged with its provenance identifier:
`[{doc_id}#{chunk_seq}] {text}`
joined by newlines.
Example: `"[doc-2#1] Production router failed due to memory exhaustion under spike load."`

In [ ]:
# YOUR CODE HERE (stuck? hint(7), then answer(7))
...

if "build_grounded_context" in locals() and callable(build_grounded_context):
    try:
        ctx = build_grounded_context([2, 4, 0])
        print(f"Grounded context:\n{ctx}")
    except Exception:
        ctx = ""

In [ ]:
check(7, lambda: "[doc-2#1]" in build_grounded_context([2, 4, 0]) and "Production router failed" in build_grounded_context([2, 4, 0]), "grounded context includes chunk text with strict document citation brackets")

## 8. Automated grounding evaluation (hallucination detection)
To verify the copilot doesn't hallucinate unsupported facts, evaluate grounding mathematically:
Write `compute_grounding_ratio(answer_sentence, context_text)` that:
1. Normalizes `answer_sentence` and `context_text` to lowercase and strips punctuation.
2. Extracts words with length $> 3$ from the answer (`ans_tokens`).
3. Extracts words with length $> 3$ from the context (`ctx_tokens`).
4. Computes the fraction of answer tokens that appear in the context:
   $\text{ratio} = \frac{|\text{ans\_tokens} \cap \text{ctx\_tokens}|}{|\text{ans\_tokens}|}$
Returns float (or `0.0` if `ans_tokens` is empty).

In [ ]:
grounded_ans = "The production router failed because memory was exhausted under spike load [doc-2#1]."
hallucinated_ans = "The quantum neural database was hacked by unauthorized alien actors."
sample_ctx = "[doc-2#1] Production router failed due to memory exhaustion under spike load."

# YOUR CODE HERE (stuck? hint(8), then answer(8))
...

if "compute_grounding_ratio" in locals() and callable(compute_grounding_ratio):
    try:
        g_score = compute_grounding_ratio(grounded_ans, sample_ctx)
        h_score = compute_grounding_ratio(hallucinated_ans, sample_ctx)
        print(f"Grounded answer score: {g_score:.2f} · Hallucinated answer score: {h_score:.2f}")
    except Exception:
        pass

In [ ]:
check(8, lambda: compute_grounding_ratio(grounded_ans, sample_ctx) >= 0.75 and compute_grounding_ratio(hallucinated_ans, sample_ctx) < 0.15, "grounding evaluator verifies cited statements and rejects ungrounded hallucinations")

## 9. Production FastAPI service handler
Write `copilot_service(request_dict)` simulating a FastAPI endpoint handler:
`request_dict` contains `{"query": str, "metadata": [word_count, is_incident, hour]}`.
1. Start timer `t0 = time.perf_counter()`.
2. Score urgency using `clf.predict_proba([request_dict["metadata"]])[0, 1]`. Mark `is_urgent = bool(p_urgent >= best_threshold)`.
3. Retrieve top-2 chunk indices with `retrieve_top_k(query, top_k=2)`.
4. Build context and citations list `[f"{chunks_data[i][1]}#{chunks_data[i][4]}" for i in hits]`.
5. Measure latency in milliseconds: `latency_ms = (time.perf_counter() - t0) * 1000`.
Return a dictionary:
`{"citations": list[str], "is_urgent": bool, "urgency_score": float, "latency_ms": float}`

In [ ]:
# YOUR CODE HERE (stuck? hint(9), then answer(9))
...

req = {"query": "router load spike", "metadata": [10, 1, 9]}
if "copilot_service" in locals() and callable(copilot_service):
    try:
        res = copilot_service(req)
        print(f"Service Response: {res}")
    except Exception:
        res = None

In [ ]:
check(9, lambda: copilot_service({"query": "router load spike", "metadata": [10, 1, 9]})["is_urgent"] is True and "doc-2#1" in copilot_service({"query": "router load spike", "metadata": [10, 1, 9]})["citations"] and copilot_service({"query": "router load spike", "metadata": [10, 1, 9]})["latency_ms"] < 100, "production service completes classification, vector retrieval, and provenance formatting in under 100 ms")

## 10. Real-time MLOps drift monitor: Population Stability Index (PSI)
In production, query distributions drift over time (e.g. shifts from routine ML questions to outage incidents).
Implement `calculate_psi(baseline_scores, production_scores, num_bins=4)`:
1. Compute empirical histograms using `np.histogram` across range `(0, 1)`.
2. Add small epsilon `1e-4` to prevent division by zero or $\log(0)$.
3. Normalize into probability distributions $P$ and $Q$.
4. Return:
   $\text{PSI} = \sum_{b=1}^B (P_b - Q_b) \cdot \ln(P_b / Q_b)$

In [ ]:
baseline_traffic = np.array([0.05, 0.12, 0.18, 0.08, 0.15, 0.22, 0.09, 0.14])
in_distribution_batch = np.array([0.08, 0.14, 0.16, 0.11, 0.17, 0.20, 0.10, 0.15])
drifted_incident_batch = np.array([0.85, 0.92, 0.78, 0.88, 0.95, 0.81, 0.90, 0.86])

# YOUR CODE HERE (stuck? hint(10), then answer(10))
...

if "calculate_psi" in locals() and callable(calculate_psi):
    try:
        stable_psi = calculate_psi(baseline_traffic, in_distribution_batch)
        drift_psi = calculate_psi(baseline_traffic, drifted_incident_batch)
        print(f"Normal traffic PSI: {stable_psi:.3f} (stable) · Incident surge PSI: {drift_psi:.3f} (alert)")
    except Exception:
        pass

In [ ]:
check(10, lambda: calculate_psi(baseline_traffic, in_distribution_batch) < 0.10 and calculate_psi(baseline_traffic, drifted_incident_batch) > 0.25, "PSI drift monitor detects distribution drift when incoming query patterns deviate from training baseline")